# Part 2. Joining HOLC GRADE hex polygons to Ebird.  
This is where we'll join HOLC grade polygons to our grid. We'll be using Duckdb to save time and space. After this we will join it with our cleaned ebird shapefile.   

For this, it is advisable to have more than 32 GB of Memory to run some of these chunks, especially with the large ebird data (HPC or Cluster)

Before this we also need to join HOLC ids to the grids.  
There should be a shapefile of 100m Hexagons that are spaced 250 m from each other (to avoid spatial biases). 

Originally we also had other hexagon geojsons: 
- 5 m  
- 10 m 
- 30 m 
- 50 m 
- 100 m *This is the one we'll be mostly using
- 250 m 
- 500 m 

But for our purposes, we will focus on using 100 m spaced hexagon file. 

For this section we will need: 

1. 100m spaced hexagon file 
2. Filtered ebird dataset from P1. 
3. holc shapefile
   

*Be sure to set up virtual environment + python interpreter (we're using Python 13+ and sf_env)

## Setting up

*You can ignore below if you already set up your virtual env and kernel.* 

1. First set up virtual env 

-  Open terminal, make sure to cd to project folder
-  Create the environment in terminal

```
conda create -n [INSERT_ENV_NAME] 
conda activate [INSERT_ENV_NAME] #activate your venv 
```

You should see in (INSERT_ENV_NAME) at the very beginning of your terminal command folder path

2. Install ipykernel 

```
conda install -n my_project ipykernel
```

Remember to always activate the correct environment before you install packages or before you run your Jupyter notebooks in VS Code!  Good luck!


## METHODS
1. Spatial join HOLC_grades to grids (100m) and then to ebird. Be sure each grid contains a unique ID 
2. Aggregate all parquets and then create new joined file for each unique ID 
3. Filters that remove hexagons: 

* NA in observations 
* They are spaced 250m apart at least (following more immediate habitat variates. This will be accomplished in P3).
* Have at least 80% in ONE HOLC grade type (includes hexagons that overlap with other HOLC grades or are outside of HOLC grades)

In [1]:
#Install and Load required packages in the notebook's environment if not already installed.  This is a one-time step.
#%conda install ipykernel
#conda install -n sf_env ipykernel --update-deps --force-reinstall
#sf_env is the name of the conda environment
#%conda install duckdb pandas geopandas pyarrow
#%conda install shapely  
#Important for geometric operations

import duckdb
import pandas as pd
import geopandas as gpd
from pathlib import Path
from shapely.geometry import Point
#Connect to DuckDB (in-memory)

con = duckdb.connect()
#con  # Optional: Print the connection object

# Make sure your DuckDB connection has spatial capabilities
con.execute("INSTALL spatial")  # `INSTALL` can be skipped if already done
con.execute("LOAD spatial")

# If you'll be working with JSON files later
con.execute("INSTALL json")
con.execute("LOAD json")
con.execute("SELECT * FROM duckdb_extensions();")

In [2]:
import os 
os.getcwd()
os.chdir("/home/oliver/Documents/redlining/")

In [3]:
#read in shapfiles and holc data 
#we should have a cleaned version of SF holc data frm SF_HOLC_cleaning code 

holc = gpd.read_file("/home/oliver/Documents/redlining/data/SF/dependent_var/SF_HOLC_cleaned.geojson")
# Create a DuckDB table with geometry, setting the correct SRID
con.execute("CREATE OR REPLACE TABLE holc AS SELECT * FROM ST_READ('/home/oliver/Documents/redlining/data/SF/dependent_var/SF_HOLC_cleaned.geojson')")

In [4]:
# Read and reproject the data 
sf_10 = gpd.read_file("/home/oliver/gdrive/Chapter1_SF_redlining_Rprj/data/dependent_var/sf_hex/sf_10.shp")
sf_100 = gpd.read_file("/home/oliver/gdrive/Chapter1_SF_redlining_Rprj/data/dependent_var/sf_hex/sf_100.shp")
sf_250 = gpd.read_file("/home/oliver/gdrive/Chapter1_SF_redlining_Rprj/data/dependent_var/sf_hex/sf_250.shp")
sf_500 = gpd.read_file("/home/oliver/gdrive/Chapter1_SF_redlining_Rprj/data/dependent_var/sf_hex/sf_500.shp") #The largest scale we could probably do is 500 m since 100 might be too large for some of the smaller HOLC grades

holc_prj = holc.to_crs(sf_100.crs)

# Group all hexagon grids together so we can loop over them
hex_groups = {"sf_10": sf_10, "sf_100": sf_100, "sf_250": sf_250, "sf_500": sf_500}


# Verify input data
print(f"HOLC CRS: {holc_prj.crs}")
print(f"Input geometry type: {holc_prj.geom_type.unique()}")


# Create temporary pandas DataFrame with explicitly selected columns
# make_valid() repairs invalid polygons instead of dropping them
df = pd.DataFrame({
    'label': holc_prj['label'],
    'grade': holc_prj['grade'],
    'geom': holc_prj.geometry.make_valid().to_wkb()
})

# Drop existing table if it exists
con.execute("DROP TABLE IF EXISTS holc_prj")

# Create table from DataFrame
con.register("holc_df", df)
con.execute("""
    CREATE TABLE holc_prj AS
    SELECT label, grade, ST_GeomFromWKB(geom) AS geom
    FROM holc_df
    WHERE geom IS NOT NULL
""")
con.unregister("holc_df")
print(f"HOLC polygons loaded: {con.execute('SELECT COUNT(*) FROM holc_prj').fetchone()[0]}")


# ============================================================================
# Join ALL hexagons in every grid to the HOLC polygons they overlap
# One row per hexagon x HOLC polygon. Hexagons outside all HOLC polygons are kept
# with NULL grade (LEFT JOIN). share_of_hex = overlap area / hexagon area.
# ============================================================================
hex_holc = {}   # e.g. hex_holc["sf_100"] is a DataFrame of the join

for name, hexes in hex_groups.items():
    print(f"Joining {name} ({len(hexes):,} hexagons)...")
    hexes = hexes.to_crs(holc_prj.crs)

    hex_df = pd.DataFrame({"hex_idx": hexes.index, "geom": hexes.geometry.to_wkb()})
    con.register("hex_df", hex_df)
    con.execute(f"""
        CREATE OR REPLACE TABLE {name} AS
        SELECT hex_idx, ST_GeomFromWKB(geom) AS geom FROM hex_df
    """)
    con.unregister("hex_df")

    con.execute(f"""
        CREATE OR REPLACE TABLE {name}_holc AS
        SELECT h.hex_idx,
               p.label,
               p.grade,
               ST_Area(ST_Intersection(h.geom, p.geom))                  AS overlap_area,
               ST_Area(ST_Intersection(h.geom, p.geom)) / ST_Area(h.geom) AS share_of_hex
        FROM {name} h
        LEFT JOIN holc_prj p ON ST_Intersects(h.geom, p.geom)
    """)

    hex_holc[name] = con.execute(f"SELECT * FROM {name}_holc").df()
    n_in = hex_holc[name].loc[hex_holc[name]["grade"].notna(), "hex_idx"].nunique()
    print(f"  {n_in:,} of {len(hexes):,} hexagons touch at least one HOLC polygon")

# Summary: number of hexagons touching each grade, per grid
summary_all = pd.DataFrame({
    name: d.dropna(subset=["grade"]).groupby("grade")["hex_idx"].nunique()
    for name, d in hex_holc.items()
}).fillna(0).astype(int)
print("\nHexagons overlapping each HOLC grade (a hexagon can count toward more than one grade):")
print(summary_all)

HOLC CRS: EPSG:32610
Input geometry type: <ArrowStringArray>
['MultiPolygon']
Length: 1, dtype: str
HOLC polygons loaded: 97
Joining sf_10 (693,846 hexagons)...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  693,846 of 693,846 hexagons touch at least one HOLC polygon
Joining sf_100 (7,983 hexagons)...
  7,983 of 7,983 hexagons touch at least one HOLC polygon
Joining sf_250 (1,450 hexagons)...
  1,450 of 1,450 hexagons touch at least one HOLC polygon
Joining sf_500 (416 hexagons)...
  416 of 416 hexagons touch at least one HOLC polygon

Hexagons overlapping each HOLC grade (a hexagon can count toward more than one grade):
        sf_10  sf_100  sf_250  sf_500
grade                                
A       53123     699     151      51
B      211263    2588     540     183
C      223293    2907     619     214
D      215036    2716     558     181


In [6]:
# Read and reproject the data 
sf_10 = gpd.read_file("/home/oliver/gdrive/Chapter1_SF_redlining_Rprj/data/dependent_var/sf_hex/sf_10.shp")
sf_100 = gpd.read_file("/home/oliver/gdrive/Chapter1_SF_redlining_Rprj/data/dependent_var/sf_hex/sf_100.shp")
sf_250 = gpd.read_file("/home/oliver/gdrive/Chapter1_SF_redlining_Rprj/data/dependent_var/sf_hex/sf_250.shp")
sf_500 = gpd.read_file("/home/oliver/gdrive/Chapter1_SF_redlining_Rprj/data/dependent_var/sf_hex/sf_500.shp") #The largest scale we could probably do is 500 m since 100 might be too large for some of the smaller HOLC grades
holc_prj = holc.to_crs(sf_100.crs)

# Group all hexagon grids together so we can loop over them
hex_groups = {"sf_10": sf_10, "sf_100": sf_100, "sf_250": sf_250, "sf_500": sf_500}


# Verify input data
print(f"HOLC CRS: {holc_prj.crs}")
print(f"Input geometry type: {holc_prj.geom_type.unique()}")


# Create temporary pandas DataFrame with explicitly selected columns
# make_valid() repairs invalid polygons instead of dropping them
df = pd.DataFrame({
    'label': holc_prj['label'],
    'grade': holc_prj['grade'],
    'geom': holc_prj.geometry.make_valid().to_wkb()
})

# Drop existing table if it exists
con.execute("DROP TABLE IF EXISTS holc_prj")

# Create table from DataFrame
con.register("holc_df", df)
con.execute("""
    CREATE TABLE holc_prj AS
    SELECT label, grade, ST_GeomFromWKB(geom) AS geom
    FROM holc_df
    WHERE geom IS NOT NULL
""")
con.unregister("holc_df")
print(f"HOLC polygons loaded: {con.execute('SELECT COUNT(*) FROM holc_prj').fetchone()[0]}")


# ============================================================================
# Join ALL hexagons in every grid to the HOLC polygons they overlap
# One row per hexagon x HOLC polygon. Hexagons outside all HOLC polygons are kept
# with NULL grade (LEFT JOIN). share_of_hex = overlap area / hexagon area.
# ============================================================================
hex_holc = {}   # e.g. hex_holc["sf_100"] is a DataFrame of the join

for name, hexes in hex_groups.items():
    print(f"Joining {name} ({len(hexes):,} hexagons)...")
    hexes = hexes.to_crs(holc_prj.crs)

    hex_df = pd.DataFrame({"hex_idx": hexes.index, "geom": hexes.geometry.to_wkb()})
    con.register("hex_df", hex_df)
    con.execute(f"""
        CREATE OR REPLACE TABLE {name} AS
        SELECT hex_idx, ST_GeomFromWKB(geom) AS geom FROM hex_df
    """)
    con.unregister("hex_df")

    con.execute(f"""
        CREATE OR REPLACE TABLE {name}_holc AS
        SELECT h.hex_idx,
               p.label,
               p.grade,
               ST_Area(ST_Intersection(h.geom, p.geom))                  AS overlap_area,
               ST_Area(ST_Intersection(h.geom, p.geom)) / ST_Area(h.geom) AS share_of_hex
        FROM {name} h
        LEFT JOIN holc_prj p ON ST_Intersects(h.geom, p.geom)
    """)

    hex_holc[name] = con.execute(f"SELECT * FROM {name}_holc").df()
    n_in = hex_holc[name].loc[hex_holc[name]["grade"].notna(), "hex_idx"].nunique()
    print(f"  {n_in:,} of {len(hexes):,} hexagons touch at least one HOLC polygon")

# Summary: number of hexagons touching each grade, per grid
summary_all = pd.DataFrame({
    name: d.dropna(subset=["grade"]).groupby("grade")["hex_idx"].nunique()
    for name, d in hex_holc.items()
}).fillna(0).astype(int) 
print("\nHexagons overlapping each HOLC grade (a hexagon can count toward more than one grade):")
print(summary_all)


HOLC CRS: EPSG:32610
Input geometry type: <ArrowStringArray>
['MultiPolygon']
Length: 1, dtype: str
HOLC polygons loaded: 97
Joining sf_10 (693,846 hexagons)...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  693,846 of 693,846 hexagons touch at least one HOLC polygon
Joining sf_100 (7,983 hexagons)...
  7,983 of 7,983 hexagons touch at least one HOLC polygon
Joining sf_250 (1,450 hexagons)...
  1,450 of 1,450 hexagons touch at least one HOLC polygon
Joining sf_500 (416 hexagons)...
  416 of 416 hexagons touch at least one HOLC polygon

Hexagons overlapping each HOLC grade (a hexagon can count toward more than one grade):
        sf_10  sf_100  sf_250  sf_500
grade                                
A       53123     699     151      51
B      211263    2588     540     183
C      223293    2907     619     214
D      215036    2716     558     181


In [9]:
sf_250.explore()
#it should look like it's a whole grid covering redlined areas that is tesellated


In [10]:
# Verify input data is valid
print(f"HOLC CRS: {holc_prj.crs}")
print(f"SF_10_CRS: {sf_10.crs}")
print(f"SF_100 CRS: {sf_100.crs}")
print(f"SF_250 CRS: {sf_250.crs}")
print(f"SF_500 CRS: {sf_500.crs}")

print(f"HOLC geometry type: {holc_prj.geom_type.unique()}")
print(f"SF_100 geometry type: {sf_100.geom_type.unique()}")


HOLC CRS: EPSG:32610
SF_10_CRS: EPSG:32610
SF_100 CRS: EPSG:32610
SF_250 CRS: EPSG:32610
SF_500 CRS: EPSG:32610
HOLC geometry type: <ArrowStringArray>
['MultiPolygon']
Length: 1, dtype: str
SF_100 geometry type: <ArrowStringArray>
['Polygon']
Length: 1, dtype: str


In [ ]:
# ============================================================================
# Hexagons with >= 75% of their area in a single dominant HOLC grade
# ============================================================================
assert sf_100.crs.is_projected, "Need a projected CRS (meters) for area calcs"

THRESH = 0.75 # we end up with around 6k hexagons

hex_ = sf_100[["geometry"]].copy()
hex_["hex_idx"] = sf_100.index
hex_["hex_area"] = hex_.area

# Merge HOLC polygons by grade so neighbouring polygons with the same grade count together
holc_by_grade = holc_prj[["grade", "geometry"]].dissolve(by="grade").reset_index()

# Intersect hexagons with grades, then sum the overlap area per hexagon per grade
pieces = gpd.overlay(hex_, holc_by_grade, how="intersection", keep_geom_type=True)
pieces["piece_area"] = pieces.area

by_grade = (pieces.groupby(["hex_idx", "grade"], as_index=False)
                  .agg(piece_area=("piece_area", "sum"), hex_area=("hex_area", "first")))
by_grade["share_of_hex"] = by_grade["piece_area"] / by_grade["hex_area"]
# Share of only the HOLC-covered part of the hexagon (ignores area outside any grade)
by_grade["share_of_covered"] = (by_grade["piece_area"] /
                                by_grade.groupby("hex_idx")["piece_area"].transform("sum"))

# Keep the dominant grade for each hexagon
dominant = (by_grade.sort_values("piece_area", ascending=False)
                    .drop_duplicates("hex_idx"))

#>= 80% of the WHOLE hexagon is one grade
dom = dominant[dominant["share_of_hex"] >= THRESH]


summary = pd.DataFrame({
    #"completely_within": joined_data["grade"].value_counts(),
    ">=75% of hex":       dom["grade"].value_counts(),
}).fillna(0).astype(int).sort_index()
summary.loc["Total"] = summary.sum()
print(summary)

# GeoDataFrame 
joined_data_75 = sf_100.join(dom.set_index("hex_idx")[["grade", "share_of_hex"]], how="inner")


       >=75% of hex
grade              
A               436
B              1873
C              1879
D              1869
Total          6057


In [17]:
joined_data_75

,id,geometry,grade,share_of_hex
0,299,"POLYGON ((543295.991 4180855.174, 543245.991 4...",B,1.000000
1,300,"POLYGON ((543295.991 4181028.379, 543245.991 4...",B,1.000000
2,301,"POLYGON ((543295.991 4181201.584, 543245.991 4...",B,1.000000
3,302,"POLYGON ((543295.991 4181374.79, 543245.991 41...",B,1.000000
4,362,"POLYGON ((543345.991 4180595.367, 543295.991 4...",B,1.000000
...,...,...,...,...
7961,15757,"POLYGON ((555395.991 4175659.022, 555345.991 4...",D,0.782901
7964,15885,"POLYGON ((555495.991 4175659.022, 555445.991 4...",D,0.973516
7967,15951,"POLYGON ((555545.991 4175918.829, 555495.991 4...",D,0.930126
7973,16270,"POLYGON ((555795.991 4175832.227, 555745.991 4...",D,0.867303


In [12]:
# ============================================================================
# Hexagons with >= THRESH of their area in a single dominant HOLC grade, for several grid sizes
# ============================================================================

#This is the result we got: 
"""
Processing sf_10...
  693,846 hexagons -> 673,448 kept
Processing sf_100...
  7,983 hexagons -> 6,057 kept
Processing sf_250...
  1,450 hexagons -> 790 kept
Processing sf_500...
  416 hexagons -> 158 kept

Hexagons with >= 75% of area in one grade:
        sf_10  sf_100  sf_250  sf_500
grade                                
A       50485     436      58      11
B      203877    1873     248      59
C      212774    1879     236      42
D      206312    1869     248      46
Total  673448    6057     790     158
"""

def filter_dominant_grade(hexes, holc, thresh=0.75):
    """Keep hexagons where >= thresh of the hexagon's area is one HOLC grade."""
    holc = holc.to_crs(hexes.crs)
    assert hexes.crs.is_projected, "Need a projected CRS (meters) for area calcs"

    hex_ = hexes[["geometry"]].copy()
    hex_["hex_idx"] = hexes.index
    hex_["hex_area"] = hex_.area

    holc_by_grade = holc[["grade", "geometry"]].dissolve(by="grade").reset_index()

    pieces = gpd.overlay(hex_, holc_by_grade, how="intersection", keep_geom_type=True)
    pieces["piece_area"] = pieces.area

    by_grade = (pieces.groupby(["hex_idx", "grade"], as_index=False)
                      .agg(piece_area=("piece_area", "sum"), hex_area=("hex_area", "first")))
    by_grade["share_of_hex"] = by_grade["piece_area"] / by_grade["hex_area"]

    dominant = (by_grade.sort_values("piece_area", ascending=False)
                        .drop_duplicates("hex_idx"))
    dom = dominant[dominant["share_of_hex"] >= thresh]

    return hexes.join(dom.set_index("hex_idx")[["grade", "share_of_hex"]], how="inner")


THRESH = 0.75
hex_files = {"sf_10": sf_10, "sf_100": sf_100, "sf_250": sf_250, "sf_500": sf_500}


results = {}   # e.g. results["sf_250"] is the filtered GeoDataFrame
counts  = {}

for name, hexes in hex_files.items():
    print(f"Processing {name}...")
    out = filter_dominant_grade(hexes, holc_prj, thresh=THRESH)

    out = out.reset_index(drop=True)
    out["FID"] = out.index + 1
    out["hex_id"] = out["FID"].astype(str) + "_" + out["grade"].astype(str)

    results[name] = out
    counts[name] = out["grade"].value_counts()
    print(f"  {len(hexes):,} hexagons -> {len(out):,} kept")

summary = pd.DataFrame(counts).fillna(0).astype(int).sort_index()
summary.loc["Total"] = summary.sum()
print(f"\nHexagons with >= {THRESH:.0%} of area in one grade:")
print(summary)


Processing sf_10...
  693,846 hexagons -> 673,448 kept
Processing sf_100...
  7,983 hexagons -> 6,057 kept
Processing sf_250...
  1,450 hexagons -> 790 kept
Processing sf_500...
  416 hexagons -> 158 kept

Hexagons with >= 75% of area in one grade:
        sf_10  sf_100  sf_250  sf_500
grade                                
A       50485     436      58      11
B      203877    1873     248      59
C      212774    1879     236      42
D      206312    1869     248      46
Total  673448    6057     790     158


# Assign FID 

In [13]:
def assign_fid(hexes, name=""):
    """Reset index, add FID (starting at 1) and hex_id, then check that IDs are unique."""
    hexes = hexes.reset_index(drop=True)
    hexes["FID"] = hexes.index + 1
    hexes["hex_id"] = hexes["FID"].astype(str) + "_" + hexes["grade"].astype(str)

    print(f"=== {name} ===")
    print(hexes.head())
    print(f"Total hexagons: {len(hexes)}")
    print(f"Unique FIDs: {hexes['FID'].nunique()}")
    print(f"All unique: {hexes['FID'].nunique() == len(hexes)}\n")
    return hexes

for name, hexes in results.items():
    results[name] = assign_fid(hexes, name)
    results[name].to_file(f"./data/SF/{name}_hex_join.geojson", driver="GeoJSON")



=== sf_10 ===
   FID                                           geometry grade  share_of_hex  \
0    1  POLYGON ((543155.991 4180993.738, 543150.991 4...     B           1.0   
1    2  POLYGON ((543155.991 4181011.059, 543150.991 4...     B           1.0   
2    3  POLYGON ((543155.991 4181028.379, 543150.991 4...     B           1.0   
3    4  POLYGON ((543155.991 4181045.7, 543150.991 418...     B           1.0   
4    5  POLYGON ((543155.991 4181063.02, 543150.991 41...     B           1.0   

  hex_id  
0    1_B  
1    2_B  
2    3_B  
3    4_B  
4    5_B  
Total hexagons: 673448
Unique FIDs: 673448
All unique: True

=== sf_100 ===
    id                                           geometry grade  share_of_hex  \
0  299  POLYGON ((543295.991 4180855.174, 543245.991 4...     B           1.0   
1  300  POLYGON ((543295.991 4181028.379, 543245.991 4...     B           1.0   
2  301  POLYGON ((543295.991 4181201.584, 543245.991 4...     B           1.0   
3  302  POLYGON ((543295.991 4181

In [ ]:
#just join for sf_100 

def assign_fid(hexes, name=""):
    """Reset index, add FID (starting at 1) and hex_id, then check that IDs are unique."""
    hexes = hexes.reset_index(drop=True)
    hexes["FID"] = hexes.index + 1
    hexes["hex_id"] = hexes["FID"].astype(str) + "_" + hexes["grade"].astype(str)

    print(f"=== {name} ===")
    print(hexes.head())
    print(f"Total hexagons: {len(hexes)}")
    print(f"Unique FIDs: {hexes['FID'].nunique()}")
    print(f"All unique: {hexes['FID'].nunique() == len(hexes)}\n")
    return hexes

joined_data_75 = assign_fid(joined_data_75) 



===  ===
    id                                           geometry grade  share_of_hex  \
0  299  POLYGON ((543295.991 4180855.174, 543245.991 4...     B           1.0   
1  300  POLYGON ((543295.991 4181028.379, 543245.991 4...     B           1.0   
2  301  POLYGON ((543295.991 4181201.584, 543245.991 4...     B           1.0   
3  302  POLYGON ((543295.991 4181374.79, 543245.991 41...     B           1.0   
4  362  POLYGON ((543345.991 4180595.367, 543295.991 4...     B           1.0   

   FID hex_id  
0    1    1_B  
1    2    2_B  
2    3    3_B  
3    4    4_B  
4    5    5_B  
Total hexagons: 6057
Unique FIDs: 6057
All unique: True



# Load in eBird for joining

In [5]:
import duckdb, os
from datetime import datetime

out_dir  = "/home/oliver/Documents/data/output"
pq_path  = f"{out_dir}/zf_SF_stat_trav_area_2010_2025.parquet"
temp_dir = f"{out_dir}/duckdb_tmp"
os.makedirs(temp_dir, exist_ok=True)

# On-disk database: tables live in the file, not in RAM
con = duckdb.connect(f"{out_dir}/ebird_sf.duckdb")
con.execute("INSTALL spatial; LOAD spatial;")
con.execute("SET memory_limit = '8GB';")              # stay well under your free RAM
con.execute("SET threads = 4;")                       # each thread needs memory; 4 is a good balance
con.execute("SET preserve_insertion_order = false;")  # lets large writes stream instead of buffering
con.execute(f"SET temp_directory = '{temp_dir}';")
con.execute("SET max_temp_directory_size = '100GB';")

# 1. A VIEW, not a table, so the parquet is read when queried and never copied
con.execute(f"CREATE OR REPLACE VIEW ebird AS SELECT * FROM read_parquet('{pq_path}');")
print(f"Total rows: {con.execute('SELECT COUNT(*) FROM ebird').fetchone()[0]:,}")

# 2. Transform UNIQUE locations only (thousands of rows, not tens of millions)
start = datetime.now()
con.execute("""
    CREATE OR REPLACE TABLE locations AS
    SELECT locality_id, latitude, longitude,
           ST_Transform(ST_Point(longitude, latitude),
                        'EPSG:4326', 'EPSG:32610', always_xy := true) AS geometry
    FROM (SELECT DISTINCT locality_id, latitude, longitude
          FROM ebird
          WHERE latitude IS NOT NULL AND longitude IS NOT NULL)
""")
n_loc = con.execute("SELECT COUNT(*) FROM locations").fetchone()[0]
print(f"{n_loc:,} unique locations transformed in {(datetime.now()-start).total_seconds():.1f}s")

# Check: UTM 10N around SF should be x ≈ 540–560k, y ≈ 4.17–4.19M
print(con.execute("""
    SELECT MIN(ST_X(geometry)) min_x, MAX(ST_X(geometry)) max_x,
           MIN(ST_Y(geometry)) min_y, MAX(ST_Y(geometry)) max_y
    FROM locations
""").df())


Total rows: 13,959,495
3,974 unique locations transformed in 0.2s
           min_x         max_x         min_y         max_y
0  415273.254158  558988.88996  4.114169e+06  4.190716e+06


In [ ]:
# Join eBird locations to EVERY hexagon grid made in the "Assign FID" step
# (./data/SF/sf_10_hex_join.geojson, sf_100_..., sf_250_..., sf_500_...)
import glob, os
import geopandas as gpd

hex_files = sorted(glob.glob("/home/oliver/Documents/redlining/data/SF/sf_*_hex_join.geojson"),
                   key=lambda p: int(os.path.basename(p).split("_")[1]))     # 10, 100, 250, 500

summary = []
for path in hex_files:
    name = os.path.basename(path).replace("_hex_join.geojson", "")         # e.g. "sf_250"
    print(f"\n{'='*60}\n{name}\n{'='*60}")

    # Points in `locations` are EPSG:32610, so the hexagons must be too
    crs = gpd.read_file(path, rows=1).crs
    assert crs.to_epsg() == 32610, f"{name} is {crs}, reproject it to EPSG:32610 first"

    # 1. Load hexagons straight into DuckDB (faster than geopandas for the 10 m grid)
    #    ::GEOMETRY drops the CRS tag, which the older ebird_sf.duckdb file can't store
    con.execute(f"""
        CREATE OR REPLACE TABLE hexes_{name} AS
        SELECT hex_id, grade, geom::GEOMETRY AS geometry
        FROM ST_Read('{path}')
    """)

    # 2. Point-in-hexagon join on the unique locations only
    con.execute(f"""
        CREATE OR REPLACE TABLE locations_hex_{name} AS
        SELECT l.locality_id, l.latitude, l.longitude, h.hex_id, h.grade,
               '{name}' AS hex_scale
        FROM locations l
        JOIN hexes_{name} h ON ST_Within(l.geometry, h.geometry)
    """)

    # Checks
    chk = con.execute(f"""
        SELECT COUNT(*) AS n_locations_in_hex,
               COUNT(DISTINCT hex_id) AS n_hexes_with_data,
               COUNT(*) - COUNT(DISTINCT (locality_id, latitude, longitude)) AS n_duplicates
        FROM locations_hex_{name}
    """).df()
    print(chk)
    print(con.execute(f"""
        SELECT grade, COUNT(DISTINCT hex_id) AS n_hex, COUNT(*) AS n_locations
        FROM locations_hex_{name} GROUP BY grade ORDER BY grade
    """).df())
    summary.append(chk.assign(hex_scale=name))

    #3. (Optional) write the joined eBird records for this grid, like the single-grid version
    con.execute(f"""
        COPY (
          SELECT e.*, l.hex_id, l.grade, l.hex_scale
          FROM ebird e
          JOIN locations_hex_{name} l USING (locality_id, latitude, longitude)
        ) TO '{out_dir}/ebird_hex_joined_{name}.parquet' (FORMAT PARQUET, COMPRESSION ZSTD)
    """)

print("\nAll grids:")
print(pd.concat(summary).set_index("hex_scale"))



sf_10


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   n_locations_in_hex  n_hexes_with_data  n_duplicates
0                1171               1048             0
  grade  n_hex  n_locations
0     A     67           70
1     B    293          307
2     C    382          471
3     D    306          323


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


sf_100
   n_locations_in_hex  n_hexes_with_data  n_duplicates
0                1046                715             0
  grade  n_hex  n_locations
0     A     37           48
1     B    196          277
2     C    257          433
3     D    225          288


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


sf_250
   n_locations_in_hex  n_hexes_with_data  n_duplicates
0                 930                381             0
  grade  n_hex  n_locations
0     A     22           42
1     B    107          229
2     C    134          394
3     D    118          265


In [ ]:
""" import duckdb
import os
from datetime import datetime

#This takes about 6 minutes
print("\n" + "="*80)
print("LOADING PARQUET AND CREATING SPATIAL DATA FOR JOINING")
print("="*80 + "\n")

# Create temp directory on D drive
temp_dir = "../data/output"
os.makedirs(temp_dir, exist_ok=True)

# Initialize connection with temp directory
con = duckdb.connect(":memory:")

# Load spatial extension
print("Loading spatial extension...")
con.execute("INSTALL spatial;")
con.execute("LOAD spatial;")

# Set up configuration
con.execute("SET threads TO 1;")
con.execute("SET memory_limit TO '16GB';")
con.execute(f"SET temp_directory='{temp_dir}';")  # Set temp directory

# ========================================================================
# 1. Load Parquet file
# ========================================================================
print("Loading parquet data...")
start_time = datetime.now()

con.execute("""
    CREATE OR REPLACE TABLE ebird AS
    SELECT * FROM read_parquet('/home/oliver/Documents/data/output/zf_SF_stat_trav_area_2010_2025.parquet');
""")

parquet_load_time = (datetime.now() - start_time).total_seconds()
print(f"Parquet loaded in {parquet_load_time:.2f} seconds")

# Check the data
row_count = con.execute("SELECT COUNT(*) FROM ebird").fetchone()[0]
print(f"Total rows: {row_count:,}")

# Show sample of column names
columns = con.execute("DESCRIBE ebird").fetchall()
print(f"\nColumns in dataset: {len(columns)}")
print("First 10 columns:", [c[0] for c in columns[:10]])


# ========================================================================
# 2. Create spatial points from lat/lon AND transform to EPSG:32610
# ========================================================================
print("\nCreating spatial points and transforming to EPSG:32610...")

# First, get distinct years
years = con.execute("""
    SELECT DISTINCT YEAR(observation_date) as year 
    FROM ebird 
    ORDER BY year
""").fetchall()
years = [y[0] for y in years]

print(f"Processing {len(years)} years: {years}")

# Create spatial table year by year with transformation in one step
for idx, year in enumerate(years):
    print(f"  Processing year {year} ({idx+1}/{len(years)})...")
    
    if idx == 0:
        # Create table with first year - transform directly
        con.execute(f"""
            CREATE TABLE ebird_transformed AS
            SELECT 
                *,
                ST_Transform(
                    ST_Point(longitude, latitude),
                    'EPSG:4326',
                    'EPSG:32610',
                     always_xy := true
                ) as geometry
            FROM ebird
            WHERE YEAR(observation_date) = {year}
              AND longitude IS NOT NULL 
              AND latitude IS NOT NULL
        """)
    else:
        # Insert remaining years
        con.execute(f"""
            INSERT INTO ebird_transformed
            SELECT 
                *,
                ST_Transform(
                    ST_Point(longitude, latitude),
                    'EPSG:4326',
                    'EPSG:32610',
                     always_xy := true
                ) as geometry
            FROM ebird
            WHERE YEAR(observation_date) = {year}
              AND longitude IS NOT NULL 
              AND latitude IS NOT NULL
        """)

print("✓ Spatial points created and transformed")

# Verify the transformation worked
spatial_count = con.execute("SELECT COUNT(*) FROM ebird_transformed").fetchone()[0]
print(f"Spatial points: {spatial_count:,}")

# Check extents to ensure transformation worked
print("\nVerifying transformation:")
extents = con.execute("""
    SELECT 
        MIN(ST_X(geometry)) as min_x,
        MAX(ST_X(geometry)) as max_x,
        MIN(ST_Y(geometry)) as min_y,
        MAX(ST_Y(geometry)) as max_y
    FROM ebird_transformed
""").df()
print(extents)

# Sample to verify (should show coordinates around 1.9M, 570K)
print("\nSample transformed points:")
print(con.execute("""
    SELECT 
        latitude, longitude,
        ST_X(geometry) as x_32610,
        ST_Y(geometry) as y_32610
    FROM ebird_transformed 
    LIMIT 5
""").df()) """


LOADING PARQUET AND CREATING SPATIAL DATA FOR JOINING

Loading spatial extension...
Loading parquet data...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Parquet loaded in 24.01 seconds
Total rows: 13,959,495

Columns in dataset: 47
First 10 columns: ['checklist_id', 'last_edited_date', 'country', 'country_code', 'state', 'state_code', 'county', 'county_code', 'iba_code', 'bcr_code']

Creating spatial points and transforming to EPSG:32610...
Processing 16 years: [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
  Processing year 2010 (1/16)...
  Processing year 2011 (2/16)...
  Processing year 2012 (3/16)...
  Processing year 2013 (4/16)...
  Processing year 2014 (5/16)...
  Processing year 2015 (6/16)...
  Processing year 2016 (7/16)...
  Processing year 2017 (8/16)...
  Processing year 2018 (9/16)...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  Processing year 2019 (10/16)...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (9 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  Processing year 2025 (16/16)...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✓ Spatial points created and transformed
Spatial points: 13,959,495

Verifying transformation:
           min_x         max_x         min_y         max_y
0  415273.254158  558988.88996  4.114169e+06  4.190716e+06

Sample transformed points:
    latitude   longitude        x_32610       y_32610
0  37.775383 -122.481793  545635.802362  4.181020e+06
1  37.775383 -122.481793  545635.802362  4.181020e+06
2  37.775383 -122.481793  545635.802362  4.181020e+06
3  37.775383 -122.481793  545635.802362  4.181020e+06
4  37.775383 -122.481793  545635.802362  4.181020e+06


# Export out eBird hours for Efforts

In [ ]:
# Only grab observation_hours, lat/long of checklists, and checklist_id
con.execute("""
    CREATE TABLE ebird_hours AS
    SELECT 
        sampling_event_identifier,
        observation_date,
        effort_hours,
        geometry 
    FROM ebird_transformed
""")
print("\n✓ Created ebird_hours table with selected columns")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


✓ Created ebird_hours table with selected columns


In [ ]:
# Step 1: Remove exact duplicates
con.execute("""
    CREATE OR REPLACE TABLE ebird_hours AS
    SELECT DISTINCT
        sampling_event_identifier,
        observation_date,
        effort_hours,
        geometry 
    FROM ebird_hours
""")
print("✓ Removed duplicate rows")

# Step 2: Sum effort hours by sampling_event_identifier and observation_date
con.execute("""
    CREATE OR REPLACE TABLE ebird_hours AS
    SELECT 
        sampling_event_identifier,
        observation_date,
        SUM(effort_hours) as effort_hours,
        -- Keep one geometry per group (using ANY_VALUE or FIRST)
        ANY_VALUE(geometry) as geometry
    FROM ebird_hours
    GROUP BY sampling_event_identifier, observation_date
""")
print("✓ Aggregated effort hours by sampling_event_identifier and observation_date")

# Verify the results
stats = con.execute("""
    SELECT 
        COUNT(*) as total_rows,
        COUNT(DISTINCT sampling_event_identifier) as unique_checklists,
        COUNT(DISTINCT CONCAT sampling_event_identifier, '|', observation_date)) as unique_combinations
    FROM ebird_hours
""").fetchone()

print(f"\nFinal stats:")
print(f"Total rows: {stats[0]:,}")
print(f"Unique sampling_event_identifiers: {stats[1]:,}")
print(f"Unique sampling_event_identifier + date combinations: {stats[2]:,}")




✓ Removed duplicate rows
✓ Aggregated effort hours by checklist_id and observation_date

Final stats:
Total rows: 50,960
Unique checklist_ids: 50,960
Unique checklist_id + date combinations: 50,960


In [ ]:
#export to csv (use in check ebird_hours script)
# Export to CSV
con.execute("""
    COPY ebird_hours 
    TO './data/SF/ebird_hours_11.18.26.csv' 
    (HEADER, DELIMITER ',')
""")
print("✓ Exported to ebird_hours.csv")

✓ Exported to ebird_hours.csv


# Effort Hours per Hexagon

In [ ]:
#create another dataframe with total effort hours: 
# Configure DuckDB to use all available cores
con.execute("SET threads TO 8")
con.execute("SET memory_limit = '12GB';")
con.execute("SET max_memory = '12GB';")
con.execute("SET preserve_insertion_order = false;")

print("Creating spatial indexes...")
con.execute("CREATE INDEX IF NOT EXISTS idx_ebird_geom ON ebird_transformed USING RTREE (geometry);")

print("Running parallel spatial join with aggregation by hexagon and species...")

con.execute("""
    CREATE OR REPLACE TABLE joined_data_effort AS
    SELECT 
        h.hex_id,
        
        -- Aggregated observation data for this hexagon
        SUM(e.max_observations) as total_max_observations,
        SUM(e.effort_hours) as total_effort_hours,
        COUNT(*) as num_checklists,
        COUNT(DISTINCT e.observation_date) as num_observation_days,
        
        -- Temporal information
        MIN(e.observation_date) as first_observed,
        MAX(e.observation_date) as last_observed,
        COUNT(DISTINCT YEAR(e.observation_date)) as num_years_observed,
        
        -- Hexagon geometry (keep for spatial analysis)
        h.geom as hexagon_geom,
        
        
    FROM ebird_transformed e
    INNER JOIN joined_data h
    ON ST_Intersects(h.geom, e.geometry)
    
    GROUP BY 
        h.hex_id,
        h.geom,
""")

print("✓ Join complete!")

# Verify results
print("\n" + "="*60)
print("="*60)




Creating spatial indexes...
Running parallel spatial join with aggregation by hexagon and species...
✓ Join complete!



In [ ]:

# Sample records
print("\nSample Records (top 10 by observations):")
sample = con.execute("""
    SELECT *,
                     FROM joined_data_effort
    LIMIT 10
""").df()
print(sample)


Sample Records (top 10 by observations):
   hex_id  total_max_observations  total_effort_hours  num_checklists  \
0  2500_C                   163.0          316.816667            1571   
1  3573_D                   224.0         1935.750000            4191   
2  2607_C                   228.0          219.116667            2595   
3  1985_B                    50.0           24.933333             343   
4  3875_D                    47.0          224.300000             723   
5  3991_D                  2400.0         6864.466667           21180   
6  3626_D                   210.0          387.250000             647   
7  3960_D                    38.0          150.833333             290   
8  2311_C                     0.0            2.450000             147   
9   190_C                  4754.0        74501.866667           96994   

   num_observation_days first_observed last_observed  num_years_observed  \
0                     9     2015-02-10    2023-04-04                   3   
1 

In [ ]:
#Export out Effort Hours per Hexagon

# Export as Parquet (without geometry to save space)
print("Exporting to Parquet...")
con.execute("""
    COPY (SELECT * FROM joined_data_effort)
    TO './data/hexid_efforthours_11.11.26.csv'
    (FORMAT CSV);
""")


print("\n" + "="*80)
print("COMPLETE!")
print("="*80)



con.close()



Exporting to Parquet...
✓ Saved: ebird_holc_joined.parquet

COMPLETE!


In [ ]:
# Check spatial coverage
coverage = con.execute("""
    SELECT 
        COUNT(DISTINCT h.hex_id) as total_hexagons,
        COUNT(DISTINCT CASE WHEN e.geometry IS NOT NULL THEN h.hex_id END) as hexagons_with_birds,
        ROUND(100.0 * COUNT(DISTINCT CASE WHEN e.geometry IS NOT NULL THEN h.hex_id END) / 
              COUNT(DISTINCT h.hex_id), 2) as pct_coverage
    FROM joined_data h
    LEFT JOIN ebird_transformed e
        ON ST_Intersects(h.geom, e.geometry)
""").df()

print(coverage)

   total_hexagons  hexagons_with_birds  pct_coverage
0            4410                  742         16.83


# Create Visual of the amount of sites

In [ ]:
#Create a bar graph of the amount of sites 
import matplotlib.pyplot as plt

# Define HOLC colors
holc_colors = {
    'A': '#9FE2BF',
    'B': '#6495ED', 
    'C': '#FFBF00',
    'D': '#DE3163'
}

# Get count of unique hexagons (sites) per HOLC grade
sites_by_grade = con.execute("""
    SELECT 
        holc_grade,
        COUNT(DISTINCT hex_id) as unique_sites
    FROM joined_data_all
    WHERE holc_grade IS NOT NULL
    GROUP BY holc_grade
    ORDER BY holc_grade
""").df()

print("\nUnique sites by HOLC grade:")
print(sites_by_grade)

# Create bar graph
fig, ax = plt.subplots(figsize=(10, 6))

# Create bars with HOLC colors
bars = ax.bar(
    sites_by_grade['holc_grade'], 
    sites_by_grade['unique_sites'],
    color=[holc_colors[grade] for grade in sites_by_grade['holc_grade']],
    edgecolor='black',
    linewidth=1.2
)

# Customize the plot
ax.set_xlabel('HOLC Grade', fontsize=12, fontweight='bold')
ax.set_ylabel('Number of Unique Sites', fontsize=12, fontweight='bold')
ax.set_title('Number of Unique Hexagon Sites by HOLC Grade', fontsize=14, fontweight='bold')

# Add value labels on top of bars
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(height)}',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

# Add grid for easier reading
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

# Optional: Also show as percentage
total_sites = sites_by_grade['unique_sites'].sum()
sites_by_grade['percentage'] = (sites_by_grade['unique_sites'] / total_sites * 100).round(2)
print("\nWith percentages:")
print(sites_by_grade)

BinderException: Binder Error: Referenced column "holc_grade" not found in FROM clause!
Candidate bindings: "hex_id", "hexagon_geom", "last_observed", "total_max_observations"

LINE 6:     WHERE holc_grade IS NOT NULL
                  ^

# Stop here Move to P3